In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
WORK.mkdir(parents=True, exist_ok=True)
import os
os.environ.setdefault('JOBLIB_TEMP_FOLDER', str(WORK))
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
from contextlib import contextmanager
import gc
import math
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from netCDF4 import Dataset, num2date
from tqdm import tqdm
import joblib
from joblib import Parallel, delayed, parallel_config

warnings.filterwarnings("ignore", category=RuntimeWarning)

RESULT_DIR = RESULTS
ERA5_DIR = DATA
OUTPUT_DIR = RESULT_DIR / "backbone_metric_comparison_parallel32_gridwise"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "cnn_lstm", "cnn_transformer", "unet", "transunet",
    "st_transunet", "da_transunet", "ta_transunet",
]
MODEL_LABELS = {
    "cnn_lstm": "CNN-LSTM",
    "cnn_transformer": "CNN-Transformer",
    "unet": "U-Net",
    "transunet": "TransUNet",
    "st_transunet": "ST-TransUNet",
    "da_transunet": "DA-TransUNet",
    "ta_transunet": "TA-TransUNet",
}
LEADS = [1, 2, 3, 4, 5, 6]
VARIABLES = ["u", "v", "w"]
VARIABLE_LABELS = {"u": "u", "v": "v", "w": "ω"}
VARIABLE_UNITS = {"u": "m/s", "v": "m/s", "w": "Pa/s"}
CANONICAL_LEVELS_HPA = np.array([1000.0, 925.0, 850.0], dtype=np.float64)

def result_file(model, kind, variable, lead):
    return RESULT_DIR / f"{model}_{kind}_{variable}_{lead}_24year_changjiang.nc"

RECOVER_LEGACY_W_SCALE = True
LEGACY_W_SCALE = -0.1

N_JOBS = 32
PARALLEL_BACKEND = "loky"
METRIC_TIME_CHUNK = 512
MIN_VALID_TIME_COUNT_FOR_R = 3
RANGE_EPS = 1e-12

ALIGN_COMMON_TEST_TIMES = True
POSITIONAL_ALIGNMENT = "tail"
REQUIRE_COMPLETE_SET = False

ETS_PERCENTILE = 99.0
INPUT_TIMESTEPS = 6
TEST_SIZE = 0.20
VALID_SIZE = 0.10
RECOMPUTE_ETS_THRESHOLDS = False
ETS_THRESHOLD_CACHE = OUTPUT_DIR / "ETS_training_99pct_thresholds.csv"
ERA5_U_FILES = {
    850: ERA5_DIR / "U-component-of-wind-850hpa.nc",
    925: ERA5_DIR / "U-component-of-wind-925hpa.nc",
    1000: ERA5_DIR / "U-component-of-wind-1000hpa.nc",
}
ERA5_V_FILES = {
    850: ERA5_DIR / "V-component-of-wind-850hpa.nc",
    925: ERA5_DIR / "V-component-of-wind-925hpa.nc",
    1000: ERA5_DIR / "V-component-of-wind-1000hpa.nc",
}

METRIC_MAP_NC = OUTPUT_DIR / "Backbone_temporal_metric_maps_parallel32.nc"
ETS_MAP_NC = OUTPUT_DIR / "Backbone_ETS_gridpoint_maps_parallel32.nc"
SUPPLEMENT_XLSX = OUTPUT_DIR / "Supplementary_backbone_metrics_detailed.xlsx"
SUPPLEMENT_CSV = OUTPUT_DIR / "Supplementary_backbone_metrics_detailed.csv"
MAIN_FIG_PNG = OUTPUT_DIR / "Fig_backbone_R_RMSE.png"
MAIN_FIG_PDF = OUTPUT_DIR / "Fig_backbone_R_RMSE.pdf"
RADAR_FIG_PNG = OUTPUT_DIR / "Fig_backbone_radar_u_v_omega_scaled.png"
RADAR_FIG_PDF = OUTPUT_DIR / "Fig_backbone_radar_u_v_omega_scaled.pdf"
RADAR_SCALE_CSV = OUTPUT_DIR / "Radar_scale_factors.csv"
QC_CSV = OUTPUT_DIR / "Backbone_metric_QC.csv"
COMMON_TIME_CSV = OUTPUT_DIR / "Common_test_time_summary.csv"
FIG_DPI = 500


In [ ]:
TIME_ALIASES = ("time", "valid_time")
LEVEL_ALIASES = ("level", "pressure", "pressure_level", "isobaricinhpa", "isobaric")
LAT_ALIASES = ("latitude", "lat")
LON_ALIASES = ("longitude", "lon")

def _match_axis(dim_names, aliases):
    lower = [str(d).lower() for d in dim_names]
    for alias in aliases:
        for i, name in enumerate(lower):
            if name == alias or alias in name:
                return i
    return None

def _choose_data_variable(nc, expected_name=None):
    if expected_name is not None and expected_name in nc.variables:
        return nc.variables[expected_name]
    coord_like = {str(x).lower() for x in list(nc.dimensions.keys()) + [
        "time", "valid_time", "level", "pressure", "pressure_level",
        "latitude", "lat", "longitude", "lon"
    ]}
    candidates = [(name, var) for name, var in nc.variables.items()
                  if name.lower() not in coord_like and var.ndim >= 3]
    if len(candidates) == 1:
        return candidates[0][1]
    raise KeyError(f"无法唯一识别数据变量：expected={expected_name}, candidates={[x[0] for x in candidates]}")

def _decode_time_variable(tv):
    units = getattr(tv, "units", None)
    calendar = getattr(tv, "calendar", "standard")
    if units is None:
        return None
    try:
        dates = num2date(tv[:], units=units, calendar=calendar,
                         only_use_cftime_datetimes=False,
                         only_use_python_datetimes=False)
        return pd.to_datetime([str(x) for x in dates], errors="coerce").to_numpy(dtype="datetime64[ns]")
    except Exception:
        return None

def _read_coord_from_dim(nc, dim_name, length):
    if dim_name is not None and dim_name in nc.variables:
        try:
            return np.asarray(nc.variables[dim_name][:], dtype=np.float64)
        except Exception:
            pass
    return np.arange(length, dtype=np.float64)

def inspect_result_file(path, expected_var=None):
    path = Path(path)
    with Dataset(path, "r") as nc:
        var = _choose_data_variable(nc, expected_var)
        dims = list(var.dimensions)
        time_axis = _match_axis(dims, TIME_ALIASES)
        level_axis = _match_axis(dims, LEVEL_ALIASES)
        lat_axis = _match_axis(dims, LAT_ALIASES)
        lon_axis = _match_axis(dims, LON_ALIASES)
        if time_axis is None:
            time_axis = int(np.argmax(var.shape))
        remaining = [i for i in range(var.ndim) if i != time_axis]
        if level_axis is None:
            cand = [i for i in remaining if var.shape[i] == 3]
            if len(cand) == 1:
                level_axis = cand[0]
        if lat_axis is None or lon_axis is None:
            spatial = [i for i in remaining if i != level_axis]
            if len(spatial) == 2:
                a, b = spatial
                if var.shape[a] <= var.shape[b]:
                    lat_axis, lon_axis = a, b
                else:
                    lat_axis, lon_axis = b, a
        if None in (time_axis, level_axis, lat_axis, lon_axis):
            raise RuntimeError(f"无法识别 {path.name} 的四个轴：dims={dims}, shape={var.shape}")
        time_dim, level_dim = dims[time_axis], dims[level_axis]
        lat_dim, lon_dim = dims[lat_axis], dims[lon_axis]
        times = _decode_time_variable(nc.variables[time_dim]) if time_dim in nc.variables else None
        levels = _read_coord_from_dim(nc, level_dim, var.shape[level_axis])
        lat = _read_coord_from_dim(nc, lat_dim, var.shape[lat_axis])
        lon = _read_coord_from_dim(nc, lon_dim, var.shape[lon_axis])
        if levels.size == 3 and np.nanmax(np.abs(levels)) > 2000:
            levels = levels / 100.0
        if levels.size == 3 and not np.all([np.any(np.isclose(levels, x, atol=1.0)) for x in CANONICAL_LEVELS_HPA]):
            levels = CANONICAL_LEVELS_HPA.copy()
        return {
            "path": path, "var_name": var.name, "dims": dims, "shape": tuple(var.shape),
            "time_axis": int(time_axis), "level_axis": int(level_axis),
            "lat_axis": int(lat_axis), "lon_axis": int(lon_axis),
            "times": times, "levels": np.asarray(levels, dtype=np.float64),
            "lat": np.asarray(lat, dtype=np.float64), "lon": np.asarray(lon, dtype=np.float64),
            "time_length": int(var.shape[time_axis]),
        }

def _canonical_level_order(levels):
    levels = np.asarray(levels, dtype=float)
    order=[]
    for target in CANONICAL_LEVELS_HPA:
        idx = np.where(np.isclose(levels, target, atol=1.0))[0]
        if len(idx) != 1:
            raise RuntimeError(f"无法把 level {levels.tolist()} 映射到 {CANONICAL_LEVELS_HPA.tolist()}")
        order.append(int(idx[0]))
    return np.asarray(order, dtype=np.int64)

def _read_time_chunk_all_levels(nc, var, meta, time_indices):
    time_indices = np.asarray(time_indices, dtype=np.int64)
    indexer = [slice(None)] * var.ndim
    if time_indices.size == 1 or np.all(np.diff(time_indices) == 1):
        indexer[meta["time_axis"]] = slice(int(time_indices[0]), int(time_indices[-1]) + 1)
    else:
        indexer[meta["time_axis"]] = time_indices
    arr = np.ma.filled(var[tuple(indexer)], np.nan)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.moveaxis(arr,
                      [meta["time_axis"], meta["level_axis"], meta["lat_axis"], meta["lon_axis"]],
                      [0, 1, 2, 3])
    return arr[:, _canonical_level_order(meta["levels"]), :, :]

def _build_time_indices(meta, common_times, common_n):
    nt = int(meta["time_length"])
    if common_times is not None:
        source_times = pd.Index(np.asarray(meta["times"], dtype="datetime64[ns]"))
        idx = source_times.get_indexer(np.asarray(common_times, dtype="datetime64[ns]"))
        if np.any(idx < 0):
            raise RuntimeError(f"{meta['path'].name} 缺少公共测试时刻。")
        return idx.astype(np.int64)
    n = min(int(common_n), nt)
    if POSITIONAL_ALIGNMENT == "tail":
        return np.arange(nt - n, nt, dtype=np.int64)
    return np.arange(0, n, dtype=np.int64)

def _recover_w_if_needed(arr):
    return arr / float(LEGACY_W_SCALE) if RECOVER_LEGACY_W_SCALE else arr

@contextmanager
def tqdm_joblib(tqdm_object):
    old_callback = joblib.parallel.BatchCompletionCallBack
    class TqdmBatchCompletionCallback(old_callback):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback


In [ ]:
inventory_rows=[]
metadata_cache={}
for model in MODELS:
    for lead in LEADS:
        for variable in VARIABLES:
            for kind in ("testy", "predicty"):
                path = result_file(model, kind, variable, lead)
                row={"model":model,"model_label":MODEL_LABELS[model],"lead_h":lead,
                     "variable":variable,"kind":kind,"path":str(path),"exists":path.exists()}
                if path.exists():
                    try:
                        meta=inspect_result_file(path, f"{kind}_{variable}")
                        metadata_cache[str(path)] = meta
                        row.update({"shape":str(meta["shape"]),"time_length":meta["time_length"],
                                    "time_decodable":meta["times"] is not None and len(meta["times"])==meta["time_length"] and np.all(~pd.isna(meta["times"])),
                                    "levels":str(meta["levels"].tolist())})
                    except Exception as e:
                        row["inspect_error"] = repr(e)
                inventory_rows.append(row)

inventory_df=pd.DataFrame(inventory_rows)
missing_df=inventory_df.loc[~inventory_df["exists"]].copy()
print("缺失文件数：", len(missing_df))
if len(missing_df):
    display(missing_df[["model_label","lead_h","variable","kind","path"]])
    if REQUIRE_COMPLETE_SET:
        raise FileNotFoundError("结果文件不完整。")

reference_meta=None
for _, row in inventory_df.loc[inventory_df["exists"]].iterrows():
    reference_meta=metadata_cache.get(row["path"])
    if reference_meta is not None:
        break
if reference_meta is None:
    raise RuntimeError("没有可读取的结果 NetCDF。")
REF_LAT=reference_meta["lat"].astype(np.float64)
REF_LON=reference_meta["lon"].astype(np.float64)
NY,NX=len(REF_LAT),len(REF_LON)
print("参考网格：", NY, "×", NX)

common_times_by_lead={}
common_n_by_lead={}
summary=[]
for lead in LEADS:
    metas=[]; lengths=[]; decodable=True
    for model in MODELS:
        for variable in VARIABLES:
            for kind in ("testy","predicty"):
                path=result_file(model,kind,variable,lead)
                meta=metadata_cache.get(str(path))
                if meta is None:
                    continue
                lengths.append(meta["time_length"])
                if meta["times"] is None or len(meta["times"])!=meta["time_length"] or np.any(pd.isna(meta["times"])):
                    decodable=False
                else:
                    metas.append(meta)
    if ALIGN_COMMON_TEST_TIMES and decodable and metas:
        common=None
        for meta in metas:
            t=np.unique(np.asarray(meta["times"],dtype="datetime64[ns]"))
            common=t if common is None else np.intersect1d(common,t,assume_unique=True)
        common=np.sort(common)
        if len(common)==0:
            raise RuntimeError(f"lead={lead} h 的绝对时间交集为空。")
        common_times_by_lead[lead]=common
        common_n_by_lead[lead]=len(common)
        summary.append({"lead_h":lead,"alignment":"absolute_timestamp_intersection",
                        "common_time_count":len(common),"first_time":str(common[0]),"last_time":str(common[-1])})
    else:
        n=int(min(lengths)) if lengths else 0
        common_times_by_lead[lead]=None
        common_n_by_lead[lead]=n
        summary.append({"lead_h":lead,"alignment":f"positional_{POSITIONAL_ALIGNMENT}",
                        "common_time_count":n,"first_time":"","last_time":""})
        print(f"[WARNING] lead={lead} h 退化为 {POSITIONAL_ALIGNMENT} 对齐，共同长度={n}。")

common_time_summary_df=pd.DataFrame(summary)
common_time_summary_df.to_csv(COMMON_TIME_CSV,index=False,encoding="utf-8-sig")
display(common_time_summary_df)


In [ ]:
def _align_spatial_map_to_reference(metric_map, lat, lon):
    out=metric_map
    lat=np.asarray(lat,dtype=float); lon=np.asarray(lon,dtype=float)
    if np.allclose(lat, REF_LAT, equal_nan=True):
        pass
    elif np.allclose(lat[::-1], REF_LAT, equal_nan=True):
        out=out[:,::-1,:]
    else:
        raise RuntimeError("latitude 与参考网格不一致。")
    if np.allclose(lon, REF_LON, equal_nan=True):
        pass
    elif np.allclose(lon[::-1], REF_LON, equal_nan=True):
        out=out[:,:,::-1]
    else:
        raise RuntimeError("longitude 与参考网格不一致。")
    return out

def compute_continuous_metric_job(model, lead, variable, common_times, common_n):
    test_path=result_file(model,"testy",variable,lead)
    pred_path=result_file(model,"predicty",variable,lead)
    if not (test_path.exists() and pred_path.exists()):
        return {"status":"missing_file","model":model,"lead":lead,"variable":variable}
    try:
        tm=inspect_result_file(test_path,f"testy_{variable}")
        pm=inspect_result_file(pred_path,f"predicty_{variable}")
        ti=_build_time_indices(tm,common_times,common_n)
        pi=_build_time_indices(pm,common_times,common_n)
        if len(ti)!=len(pi) or len(ti)==0:
            raise RuntimeError(f"对齐长度异常：{len(ti)} vs {len(pi)}")
        shape=(3,len(tm["lat"]),len(tm["lon"]))
        count=np.zeros(shape,dtype=np.int64)
        sy=np.zeros(shape); sp=np.zeros(shape); sy2=np.zeros(shape); sp2=np.zeros(shape); syp=np.zeros(shape)
        sse=np.zeros(shape); sae=np.zeros(shape)
        ymin=np.full(shape,np.inf); ymax=np.full(shape,-np.inf)
        with Dataset(test_path,"r") as nct, Dataset(pred_path,"r") as ncp:
            vt=_choose_data_variable(nct,f"testy_{variable}")
            vp=_choose_data_variable(ncp,f"predicty_{variable}")
            for s in range(0,len(ti),METRIC_TIME_CHUNK):
                e=min(s+METRIC_TIME_CHUNK,len(ti))
                y=_read_time_chunk_all_levels(nct,vt,tm,ti[s:e])
                p=_read_time_chunk_all_levels(ncp,vp,pm,pi[s:e])
                if variable=="w":
                    y=_recover_w_if_needed(y); p=_recover_w_if_needed(p)
                valid=np.isfinite(y)&np.isfinite(p)
                count += valid.sum(axis=0,dtype=np.int64)
                y0=np.where(valid,y,0.0); p0=np.where(valid,p,0.0)
                sy += np.sum(y0,axis=0,dtype=np.float64)
                sp += np.sum(p0,axis=0,dtype=np.float64)
                sy2 += np.sum(y0*y0,axis=0,dtype=np.float64)
                sp2 += np.sum(p0*p0,axis=0,dtype=np.float64)
                syp += np.sum(y0*p0,axis=0,dtype=np.float64)
                d=p0-y0
                sse += np.sum(np.where(valid,d*d,0.0),axis=0,dtype=np.float64)
                sae += np.sum(np.where(valid,np.abs(d),0.0),axis=0,dtype=np.float64)
                ymin=np.minimum(ymin,np.min(np.where(valid,y,np.inf),axis=0))
                ymax=np.maximum(ymax,np.max(np.where(valid,y,-np.inf),axis=0))
        n=count.astype(np.float64)
        with np.errstate(divide="ignore",invalid="ignore"):
            rmse=np.sqrt(sse/n); mae=sae/n
            cov=syp-sy*sp/n
            vy=sy2-sy*sy/n; vpv=sp2-sp*sp/n
            denom=np.sqrt(np.maximum(vy,0)*np.maximum(vpv,0))
            R=cov/denom
            yrange=ymax-ymin
            rmspe=rmse/yrange
        rmse[count<=0]=np.nan; mae[count<=0]=np.nan
        R[(count<MIN_VALID_TIME_COUNT_FOR_R)|(~np.isfinite(denom))|(denom<=0)]=np.nan
        R=np.clip(R,-1.0,1.0)
        rmspe[(count<=0)|(~np.isfinite(yrange))|(yrange<=RANGE_EPS)]=np.nan
        maps={}
        for name,arr in {"R":R,"RMSE":rmse,"MAE":mae,"RMSPE":rmspe,"N":count.astype(np.float64)}.items():
            maps[name]=_align_spatial_map_to_reference(arr,tm["lat"],tm["lon"]).astype(np.float32)
        return {"status":"ok","model":model,"lead":lead,"variable":variable,
                "aligned_time_count":len(ti),"maps":maps}
    except Exception as e:
        return {"status":"error","model":model,"lead":lead,"variable":variable,"error":repr(e)}


In [ ]:
tasks=[(model,lead,variable,common_times_by_lead.get(lead),common_n_by_lead.get(lead,0))
       for model in MODELS for lead in LEADS for variable in VARIABLES]
print("并行 job 数：",len(tasks))
progress=tqdm(total=len(tasks),desc="连续指标并行计算",unit="job",dynamic_ncols=True)
with tqdm_joblib(progress):
    with parallel_config(backend=PARALLEL_BACKEND,n_jobs=N_JOBS,inner_max_num_threads=1):
        results=Parallel(batch_size=1,pre_dispatch=N_JOBS)(delayed(compute_continuous_metric_job)(*t) for t in tasks)
progress.close()

shape=(len(MODELS),len(LEADS),len(VARIABLES),3,NY,NX)
R_MAP=np.full(shape,np.nan,dtype=np.float32)
RMSE_MAP=np.full(shape,np.nan,dtype=np.float32)
MAE_MAP=np.full(shape,np.nan,dtype=np.float32)
RMSPE_MAP=np.full(shape,np.nan,dtype=np.float32)
N_MAP=np.zeros(shape,dtype=np.int32)
mi_map={m:i for i,m in enumerate(MODELS)}
li_map={l:i for i,l in enumerate(LEADS)}
vi_map={v:i for i,v in enumerate(VARIABLES)}
qc=[]
for r in results:
    qc.append({"model":r.get("model"),"model_label":MODEL_LABELS.get(r.get("model"),r.get("model")),
               "lead_h":r.get("lead"),"variable":r.get("variable"),"status":r.get("status"),
               "aligned_time_count":r.get("aligned_time_count",np.nan),"error":r.get("error","")})
    if r["status"]!="ok":
        continue
    mi,li,vi=mi_map[r["model"]],li_map[r["lead"]],vi_map[r["variable"]]
    R_MAP[mi,li,vi]=r["maps"]["R"]
    RMSE_MAP[mi,li,vi]=r["maps"]["RMSE"]
    MAE_MAP[mi,li,vi]=r["maps"]["MAE"]
    RMSPE_MAP[mi,li,vi]=r["maps"]["RMSPE"]
    N_MAP[mi,li,vi]=np.nan_to_num(r["maps"]["N"],nan=0).astype(np.int32)
qc_df=pd.DataFrame(qc)
qc_df.to_csv(QC_CSV,index=False,encoding="utf-8-sig")
print("成功：",int((qc_df.status=="ok").sum()),"；失败/缺失：",int((qc_df.status!="ok").sum()))
if np.any(qc_df.status!="ok"):
    display(qc_df.loc[qc_df.status!="ok"])
    if REQUIRE_COMPLETE_SET:
        raise RuntimeError("存在失败或缺失任务。")


In [ ]:
metric_ds=xr.Dataset(
    data_vars={
        "R":(("model","lead_h","variable","level_hPa","latitude","longitude"),R_MAP),
        "RMSE":(("model","lead_h","variable","level_hPa","latitude","longitude"),RMSE_MAP),
        "MAE":(("model","lead_h","variable","level_hPa","latitude","longitude"),MAE_MAP),
        "RMSPE":(("model","lead_h","variable","level_hPa","latitude","longitude"),RMSPE_MAP),
        "N_valid_time":(("model","lead_h","variable","level_hPa","latitude","longitude"),N_MAP),
    },
    coords={
        "model":[MODEL_LABELS[m] for m in MODELS],
        "lead_h":np.asarray(LEADS,dtype=np.int16),
        "variable":[VARIABLE_LABELS[v] for v in VARIABLES],
        "level_hPa":CANONICAL_LEVELS_HPA.astype(np.float32),
        "latitude":REF_LAT.astype(np.float32),"longitude":REF_LON.astype(np.float32),
    },
    attrs={"description":"R/RMSE/MAE/RMSPE were calculated only along time at every grid point.",
           "parallel_n_jobs":N_JOBS,"parallel_backend":PARALLEL_BACKEND,
           "legacy_w_recovered":str(bool(RECOVER_LEGACY_W_SCALE)),"w_unit":"Pa/s","u_v_unit":"m/s"}
)
enc={k:{"zlib":True,"complevel":9,"shuffle":True,"dtype":"float32"} for k in ["R","RMSE","MAE","RMSPE"]}
enc["N_valid_time"]={"zlib":True,"complevel":9,"shuffle":True,"dtype":"int32"}
metric_ds.to_netcdf(METRIC_MAP_NC,engine="netcdf4",encoding=enc)
print("保存逐格点指标：",METRIC_MAP_NC)

rows=[]
for mi,model in enumerate(MODELS):
    for li,lead in enumerate(LEADS):
        for vi,variable in enumerate(VARIABLES):
            for zi,level in enumerate(CANONICAL_LEVELS_HPA):
                rows.append({
                    "Model":MODEL_LABELS[model],"Lead_h":lead,"Variable":VARIABLE_LABELS[variable],
                    "Level_hPa":int(round(level)),
                    "R":float(np.nanmean(R_MAP[mi,li,vi,zi])),
                    "RMSE":float(np.nanmean(RMSE_MAP[mi,li,vi,zi])),
                    "MAE":float(np.nanmean(MAE_MAP[mi,li,vi,zi])),
                    "RMSPE":float(np.nanmean(RMSPE_MAP[mi,li,vi,zi])),
                })
supplement_table=pd.DataFrame(rows)
model_order={MODEL_LABELS[m]:i for i,m in enumerate(MODELS)}
var_order={"u":0,"v":1,"ω":2}; level_order={1000:0,925:1,850:2}
supplement_table["_m"]=supplement_table.Model.map(model_order)
supplement_table["_v"]=supplement_table.Variable.map(var_order)
supplement_table["_z"]=supplement_table.Level_hPa.map(level_order)
supplement_table=(supplement_table.sort_values(["_m","Lead_h","_v","_z"])
                  .drop(columns=["_m","_v","_z"]).reset_index(drop=True))
with pd.ExcelWriter(SUPPLEMENT_XLSX,engine="openpyxl") as writer:
    supplement_table.to_excel(writer,sheet_name="Supplement_table",index=False)
    ws=writer.book["Supplement_table"]; ws.freeze_panes="A2"; ws.auto_filter.ref=ws.dimensions
    for col in ws.columns:
        letter=col[0].column_letter
        mx=max(len(str(c.value)) if c.value is not None else 0 for c in col[:500])
        ws.column_dimensions[letter].width=min(max(mx+2,11),20)
supplement_table.to_csv(SUPPLEMENT_CSV,index=False,encoding="utf-8-sig")
print("补充材料表格行数：",len(supplement_table))
display(supplement_table.head(30))


In [ ]:
main_rows = []

for mi, model in enumerate(MODELS):
    for li, lead in enumerate(LEADS):
        main_rows.append({
            "model": model,
            "model_label": MODEL_LABELS[model],
            "lead_h": lead,

            "R_mean": float(np.nanmean(R_MAP[mi, li])),

            "RMSPE_mean": float(np.nanmean(RMSPE_MAP[mi, li])),
        })

main_summary = pd.DataFrame(main_rows)


In [ ]:
radar_rows = []

for variable in VARIABLES:
    vl = VARIABLE_LABELS[variable]

    for model in MODELS:
        sub = supplement_table.loc[
            (supplement_table.Model == MODEL_LABELS[model]) &
            (supplement_table.Variable == vl)
        ]

        r = float(np.nanmean(sub.R))

        radar_rows.append({
            "model": model,
            "model_label": MODEL_LABELS[model],
            "variable": variable,
            "variable_label": vl,

            "one_minus_R": 1.0 - r,
            "RMSE": float(np.nanmean(sub.RMSE)),
            "MAE": float(np.nanmean(sub.MAE)),
            "RMSPE": float(np.nanmean(sub.RMSPE)),
        })

radar_raw = pd.DataFrame(radar_rows)


NICE_DIVISORS = np.array([
    0.0001, 0.0002, 0.0005,
    0.001, 0.002, 0.005,
    0.01, 0.02, 0.05,
    0.1, 0.2, 0.5,
    1, 2, 5,
    10, 20, 50,
    100, 200, 500, 1000
], dtype=float)

RADAR_TARGET_MAX = 0.8


def choose_nice_divisor(values, target=RADAR_TARGET_MAX):
    """
    从 NICE_DIVISORS 中选择一个除数，
    使缩放后的最大值尽量靠近 target。
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]

    if len(v) == 0 or np.max(np.abs(v)) <= 0:
        return 1.0

    vmax = float(np.max(np.abs(v)))

    idx = int(
        np.argmin(
            np.abs(vmax / NICE_DIVISORS - target)
        )
    )

    return float(NICE_DIVISORS[idx])


scale_rows = []
scaled_rows = []

metrics = [
    "one_minus_R",
    "RMSE",
    "MAE",
    "RMSPE"
]


for variable in VARIABLES:

    sub = radar_raw.loc[
        radar_raw.variable == variable
    ].copy()

    divisors = {
        m: choose_nice_divisor(
            sub[m].to_numpy(float)
        )
        for m in metrics
    }

    for m in metrics:

        d = divisors[m]

        label = (
            "1-R"
            if m == "one_minus_R"
            else m
        )

        formula = (
            f"(1-R)/{d:g}"
            if m == "one_minus_R"
            else f"{m}/{d:g}"
        )

        scale_rows.append({
            "Variable": VARIABLE_LABELS[variable],
            "Metric": label,
            "Divisor": d,
            "Formula": formula
        })


    for _, row in sub.iterrows():

        scaled_rows.append({
            "model": row.model,
            "model_label": row.model_label,
            "variable": variable,

            "scaled_one_minus_R":
                row.one_minus_R /
                divisors["one_minus_R"],

            "scaled_RMSE":
                row.RMSE /
                divisors["RMSE"],

            "scaled_MAE":
                row.MAE /
                divisors["MAE"],

            "scaled_RMSPE":
                row.RMSPE /
                divisors["RMSPE"],
        })


radar_scale_df = pd.DataFrame(scale_rows)
radar_scaled = pd.DataFrame(scaled_rows)


radar_scale_df.to_csv(
    RADAR_SCALE_CSV,
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
import numpy as np
import pandas as pd

MAIN_BAR_EXCEL = OUTPUT_DIR / "Supplementary_backbone_R_RMSPE.xlsx"


model_order_labels = [
    MODEL_LABELS[m]
    for m in MODELS
]


R_table = (
    main_summary
    .pivot(
        index="lead_h",
        columns="model_label",
        values="R_mean"
    )
    .reindex(
        index=LEADS,
        columns=model_order_labels
    )
)

R_table.index.name = "Forecast lead time (h)"


RMSPE_table = (
    main_summary
    .pivot(
        index="lead_h",
        columns="model_label",
        values="RMSPE_mean"
    )
    .reindex(
        index=LEADS,
        columns=model_order_labels
    )
)

RMSPE_table.index.name = "Forecast lead time (h)"


best_rows = []

for lead in LEADS:

    r_values = R_table.loc[lead]
    rmspe_values = RMSPE_table.loc[lead]

    best_r = float(
        np.nanmax(
            r_values.to_numpy(float)
        )
    )

    best_rmspe = float(
        np.nanmin(
            rmspe_values.to_numpy(float)
        )
    )


    best_r_models = r_values.index[
        np.isclose(
            r_values.to_numpy(float),
            best_r,
            rtol=1e-10,
            atol=1e-12
        )
    ].tolist()

    best_rmspe_models = rmspe_values.index[
        np.isclose(
            rmspe_values.to_numpy(float),
            best_rmspe,
            rtol=1e-10,
            atol=1e-12
        )
    ].tolist()


    best_rows.append({
        "Forecast lead time (h)": lead,

        "Best R model":
            "; ".join(best_r_models),

        "Best R":
            best_r,

        "Best RMSPE model":
            "; ".join(best_rmspe_models),

        "Best RMSPE":
            best_rmspe
    })


best_table = pd.DataFrame(
    best_rows
)


long_table = (
    main_summary[
        [
            "lead_h",
            "model_label",
            "R_mean",
            "RMSPE_mean"
        ]
    ]
    .copy()
    .rename(
        columns={
            "lead_h":
                "Forecast lead time (h)",

            "model_label":
                "Model",

            "R_mean":
                "R",

            "RMSPE_mean":
                "RMSPE"
        }
    )
)


long_table["Best_R"] = False
long_table["Best_RMSPE"] = False

for lead in LEADS:

    idx = (
        long_table[
            "Forecast lead time (h)"
        ] == lead
    )

    tmp = long_table.loc[idx]


    r_best = tmp["R"].max()

    long_table.loc[
        idx &
        np.isclose(
            long_table["R"],
            r_best,
            rtol=1e-10,
            atol=1e-12
        ),
        "Best_R"
    ] = True


    rmspe_best = tmp["RMSPE"].min()

    long_table.loc[
        idx &
        np.isclose(
            long_table["RMSPE"],
            rmspe_best,
            rtol=1e-10,
            atol=1e-12
        ),
        "Best_RMSPE"
    ] = True


with pd.ExcelWriter(
    MAIN_BAR_EXCEL
) as writer:

    R_table.to_excel(
        writer,
        sheet_name="R"
    )

    RMSPE_table.to_excel(
        writer,
        sheet_name="RMSPE"
    )

    best_table.to_excel(
        writer,
        sheet_name="Best_models",
        index=False
    )

    long_table.to_excel(
        writer,
        sheet_name="Long_format",
        index=False
    )


In [ ]:
import numpy as np
import pandas as pd

COMPONENT_BAR_EXCEL = (
    OUTPUT_DIR /
    "Supplementary_backbone_u_v_omega_metrics.xlsx"
)


metric_key_order = [
    "scaled_one_minus_R",
    "scaled_RMSE",
    "scaled_MAE",
    "scaled_RMSPE"
]

metric_display_order = [
    "1-R",
    "RMSE",
    "MAE",
    "RMSPE"
]

model_order_labels = [
    MODEL_LABELS[m]
    for m in MODELS
]


scaled_tables = {}

for variable in VARIABLES:

    sub = radar_scaled.loc[
        radar_scaled.variable == variable
    ].copy()


    sf = radar_scale_df.loc[
        radar_scale_df["Variable"] ==
        VARIABLE_LABELS[variable]
    ]

    formula_map = {
        row["Metric"]: row["Formula"]
        for _, row in sf.iterrows()
    }


    rows = []

    for metric_key, metric_name in zip(
        metric_key_order,
        metric_display_order
    ):

        row_out = {
            "Metric":
                metric_name,

            "Scaled metric":
                formula_map[metric_name]
        }


        for model in MODELS:

            tmp = sub.loc[
                sub.model == model
            ]

            row_out[
                MODEL_LABELS[model]
            ] = float(
                tmp.iloc[0][metric_key]
            )


        rows.append(
            row_out
        )


    scaled_tables[variable] = pd.DataFrame(
        rows
    )


best_rows = []

for variable in VARIABLES:

    sub = radar_scaled.loc[
        radar_scaled.variable == variable
    ].copy()


    sf = radar_scale_df.loc[
        radar_scale_df["Variable"] ==
        VARIABLE_LABELS[variable]
    ]

    formula_map = {
        row["Metric"]: row["Formula"]
        for _, row in sf.iterrows()
    }


    for metric_key, metric_name in zip(
        metric_key_order,
        metric_display_order
    ):

        values = []

        for model in MODELS:

            tmp = sub.loc[
                sub.model == model
            ]

            values.append(
                float(
                    tmp.iloc[0][metric_key]
                )
            )


        values = np.asarray(
            values,
            dtype=float
        )

        best_value = float(
            np.nanmin(values)
        )

        best_indices = np.where(
            np.isclose(
                values,
                best_value,
                rtol=1e-10,
                atol=1e-12
            )
        )[0]

        best_models = [
            MODEL_LABELS[
                MODELS[i]
            ]
            for i in best_indices
        ]


        best_rows.append({
            "Variable":
                VARIABLE_LABELS[variable],

            "Metric":
                metric_name,

            "Scaled metric":
                formula_map[metric_name],

            "Best model":
                "; ".join(best_models),

            "Best scaled value":
                best_value
        })


best_component_table = pd.DataFrame(
    best_rows
)


scaled_long_rows = []

for variable in VARIABLES:

    sub = radar_scaled.loc[
        radar_scaled.variable == variable
    ].copy()


    sf = radar_scale_df.loc[
        radar_scale_df["Variable"] ==
        VARIABLE_LABELS[variable]
    ]

    formula_map = {
        row["Metric"]: row["Formula"]
        for _, row in sf.iterrows()
    }


    for metric_key, metric_name in zip(
        metric_key_order,
        metric_display_order
    ):

        metric_values = []

        for model in MODELS:

            tmp = sub.loc[
                sub.model == model
            ]

            metric_values.append(
                float(
                    tmp.iloc[0][metric_key]
                )
            )


        metric_values = np.asarray(
            metric_values,
            dtype=float
        )

        best_value = np.nanmin(
            metric_values
        )


        for i, model in enumerate(MODELS):

            value = float(
                metric_values[i]
            )

            scaled_long_rows.append({
                "Variable":
                    VARIABLE_LABELS[variable],

                "Metric":
                    metric_name,

                "Scaled metric":
                    formula_map[metric_name],

                "Model":
                    MODEL_LABELS[model],

                "Scaled value":
                    value,

                "Is_best":
                    bool(
                        np.isclose(
                            value,
                            best_value,
                            rtol=1e-10,
                            atol=1e-12
                        )
                    )
            })


scaled_long_table = pd.DataFrame(
    scaled_long_rows
)


raw_table = radar_raw.copy()

raw_table = raw_table.rename(
    columns={
        "model_label":
            "Model",

        "variable_label":
            "Variable",

        "one_minus_R":
            "1-R"
    }
)

raw_table = raw_table.drop(
    columns=[
        "model",
        "variable"
    ],
    errors="ignore"
)

raw_columns = [
    "Variable",
    "Model",
    "1-R",
    "RMSE",
    "MAE",
    "RMSPE"
]

raw_table = raw_table[
    raw_columns
]


scale_table = radar_scale_df.copy()

scale_table = scale_table[
    [
        "Variable",
        "Metric",
        "Divisor",
        "Formula"
    ]
]


with pd.ExcelWriter(
    COMPONENT_BAR_EXCEL
) as writer:


    scaled_tables["u"].to_excel(
        writer,
        sheet_name="u_scaled",
        index=False
    )

    scaled_tables["v"].to_excel(
        writer,
        sheet_name="v_scaled",
        index=False
    )

    scaled_tables["w"].to_excel(
        writer,
        sheet_name="omega_scaled",
        index=False
    )


    scale_table.to_excel(
        writer,
        sheet_name="Scale_factors",
        index=False
    )


    best_component_table.to_excel(
        writer,
        sheet_name="Best_models",
        index=False
    )


    scaled_long_table.to_excel(
        writer,
        sheet_name="Scaled_long",
        index=False
    )


    raw_table.to_excel(
        writer,
        sheet_name="Raw_metrics",
        index=False
    )


print(
    "u / v / omega 指标汇总结果已保存：",
    COMPONENT_BAR_EXCEL
)

print("\n各变量、各指标的最佳模型：")
display(best_component_table)
